In [1]:
import tensorflow as tf 
import cv2 
import numpy as np 
import os 
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers,models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet_v2 import EfficientNetV2B2, preprocess_input
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [2]:
Img_size=224
Epochs=30
Batch_size=16

# Load Classification Images

In [4]:
x=[]
y=[]
classes=["PNEUMONIA", "NORMAL"]

for idx, cls in enumerate(classes): 
    folder=f'/Users/utkarshnageshwar/pneumonia-classification-app-main/chest_xray/train/{cls}'
    for name in os.listdir(folder): 
        if name.lower().endswith(('.jpg', '.jpeg', '.png')):
            path=os.path.join(folder,name)
            img=cv2.imread(path)
            img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
            img=cv2.resize(img,(Img_size,Img_size))
            x.append(img)
            y.append(idx)

x = np.array(x)
x = preprocess_input(x.astype(np.float32))
y=to_categorical(y,2)

# Train Validate Split

In [5]:
xtrain,xval,ytrain,yval=train_test_split(x,y,test_size=0.2,random_state=42, stratify=y) 

# Data Augmentation

In [6]:
data_aug = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
)

# Handling Class Imbalance

In [7]:
yint = np.argmax(ytrain, axis=1)

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(yint),
    y=yint
)

class_weights = dict(enumerate(weights))

# Model Building with EfficientNet

In [8]:
base_model = EfficientNetV2B2(
    weights="imagenet",
    include_top=False,
    input_shape=(Img_size, Img_size, 3), 
)

base_model.trainable = True 

for layer in base_model.layers[:-40]: 
    layer.trainable = False

2026-04-02 15:08:48.137027: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2026-04-02 15:08:48.137290: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-04-02 15:08:48.137297: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-04-02 15:08:48.137951: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-02 15:08:48.138257: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [9]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1024, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.25),

    layers.Dense(2, activation='softmax')
])

In [10]:
model.compile(optimizer='adam', loss='categorical_crossentropy',metrics=['accuracy'])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 efficientnetv2-b2 (Functio  (None, 7, 7, 1408)        8769374   
 nal)                                                            
                                                                 
 global_average_pooling2d (  (None, 1408)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 dense (Dense)               (None, 1024)              1442816   
                                                                 
 batch_normalization (Batch  (None, 1024)              4096      
 Normalization)                                                  
                                                                 
 dropout (Dropout)           (None, 1024)              0         
                                                        

# Model training

In [11]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

In [12]:
model.fit(
    data_aug.flow(xtrain, ytrain, batch_size=Batch_size, shuffle=True),
    epochs=Epochs,
    batch_size=Batch_size,
    validation_data=(xval,yval),
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/30


2026-04-02 15:08:52.941202: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


261/261 [==============================] - ETA: 0s - loss: 0.4016 - accuracy: 0.8567

2026-04-02 15:09:19.617057: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


261/261 [==============================] - 35s 118ms/step - loss: 0.4016 - accuracy: 0.8567 - val_loss: 0.3723 - val_accuracy: 0.8764
Epoch 2/30
261/261 [==============================] - 28s 106ms/step - loss: 0.2957 - accuracy: 0.8948 - val_loss: 0.2074 - val_accuracy: 0.9368
Epoch 3/30
261/261 [==============================] - 32s 122ms/step - loss: 0.3223 - accuracy: 0.8840 - val_loss: 0.5937 - val_accuracy: 0.8228
Epoch 4/30
261/261 [==============================] - 61s 233ms/step - loss: 0.2494 - accuracy: 0.9034 - val_loss: 0.8358 - val_accuracy: 0.6791
Epoch 5/30
261/261 [==============================] - 46s 177ms/step - loss: 0.2122 - accuracy: 0.9163 - val_loss: 0.5729 - val_accuracy: 0.8027
Epoch 6/30
261/261 [==============================] - 48s 185ms/step - loss: 0.1828 - accuracy: 0.9317 - val_loss: 0.3874 - val_accuracy: 0.8716
Epoch 7/30
261/261 [==============================] - 49s 189ms/step - loss: 0.1822 - accuracy: 0.9305 - val_loss: 0.1141 - val_accuracy: 0.9

# Saving Model

In [13]:
model.save("../model/classifier_model.keras")